# 2a. Parameter Recovery & SBC

**Purpose**: Can we recover known parameters from synthetic data? Can we correctly
identify which model (BE vs SC) generated the data?

**Protocol**:
1. Generate synthetic animals from known BE and SC parameters
2. Recover parameters via grid-search CV and SBI
3. SBI tested with 3 stat sets: heuristics only, update matrix only, mixed
4. For each method: true vs recovered scatter, coverage, model identification accuracy

**Two modes**:
- `QUICK = True`: 5 synthetic animals, coarse grid, 5K SBI sims (~10–15 min)
- `QUICK = False`: loads pre-computed results from disk (run training separately)

---

**What this validates**:
- Parameter identifiability: which params can actually be recovered?
- Posterior calibration (SBC): are credible intervals honest?
- Model selection reliability: false positive rate for BE-vs-SC classification
- Stat set comparison: which summary stats are most informative?

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import time
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

In [ ]:
from models.BE_core import BEParams, BEState, BEModel
from models.SC_core import SCParams, SCState, SCModel
from behav_utils.data.synthetic import sample_stimuli
from behav_utils.analysis.summary_stats import compute_summary_stats
from behav_utils.analysis.update_matrix import compute_update_matrix, matrix_error
from behav_utils.plotting.styles import COLOURS, apply_style

from analysis.grid_search import grid_search_cv, COARSE_GRID, DEFAULT_GRID
from analysis.cv_utils import format_params

apply_style()

## 1. Configuration

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# MODE
# ═══════════════════════════════════════════════════════════════════════════════
QUICK = True              # True = run everything in notebook (small scale)
                           # False = load pre-computed results from RESULTS_DIR

RESULTS_DIR = Path('../results/recovery')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# SYNTHETIC DATA PARAMETERS
# ═══════════════════════════════════════════════════════════════════════════════
if QUICK:
    N_SYNTHETIC = 5           # synthetic animals per model
    N_SESSIONS = 8            # sessions per synthetic animal
    TRIALS_PER_SESSION = 300  # trials per session
    BURN_IN = 500
    N_CV_SEEDS = 4            # grid-search CV seeds
    N_SBI_SIMS = 5_000        # SBI training simulations
    N_SBI_RECOVERY = 20       # SBI recovery tests
    N_SBC_RUNS = 50           # SBC iterations
else:
    N_SYNTHETIC = 20
    N_SESSIONS = 10
    TRIALS_PER_SESSION = 300
    BURN_IN = 1000
    N_CV_SEEDS = 32
    N_SBI_SIMS = 50_000
    N_SBI_RECOVERY = 100
    N_SBC_RUNS = 500

# Stat sets for SBI
HEURISTIC_STATS = ['accuracy', 'psychometric', 'recency', 'win_stay', 'stimulus_sensitivity']
UM_STATS = ['update_matrix']
MIXED_STATS = HEURISTIC_STATS + UM_STATS

SEED = 42
print(f"Mode: {'QUICK' if QUICK else 'FULL'}")
print(f"  {N_SYNTHETIC} animals × {N_SESSIONS} sessions × {TRIALS_PER_SESSION} trials")
print(f"  Grid-search: {N_CV_SEEDS} seeds")
print(f"  SBI: {N_SBI_SIMS:,} sims, {N_SBI_RECOVERY} recovery tests, {N_SBC_RUNS} SBC runs")

## 2. Generate Synthetic Data

Sample BE and SC parameters from the prior, simulate expert-like sessions,
store the trials in SessionData-like containers.

In [ ]:
from behav_utils.data.synthetic import generate_synthetic_animal


def make_synthetic_animals(model_type, n_animals, n_sessions, trials_per_session,
                           burn_in, base_seed):
    """Generate synthetic animals with known parameters using make_simulator."""
    rng_gen = np.random.default_rng(base_seed)
    animals = []

    for i in range(n_animals):
        seed_i = base_seed + i * 100

        if model_type == 'BE':
            params = BEParams.sample_prior(rng_gen)
            sim = BEModel.make_simulator(params, burn_in=burn_in, seed=seed_i)
        else:
            params = SCParams.sample_prior(rng_gen)
            sim = SCModel.make_simulator(params, burn_in=burn_in, seed=seed_i)

        aid = f'{model_type}_synth_{i:02d}'
        animal, info = generate_synthetic_animal(
            animal_id=aid,
            n_sessions=n_sessions,
            trials_per_session=trials_per_session,
            seed=seed_i,
            simulator=sim,
            stage='Full_Task_Cont',
        )

        acc = np.mean([s.stats(['accuracy'])['accuracy'] for s in animal.sessions])

        animals.append({
            'animal_id': aid,
            'true_model': model_type,
            'true_params': params.to_dict(),
            'sessions': animal.sessions,
            'mean_accuracy': acc,
        })

    return animals

In [ ]:
synthetic_animals = []
synthetic_animals.extend(make_synthetic_animals(
    'BE', N_SYNTHETIC, N_SESSIONS, TRIALS_PER_SESSION, BURN_IN, SEED,
))
synthetic_animals.extend(make_synthetic_animals(
    'SC', N_SYNTHETIC, N_SESSIONS, TRIALS_PER_SESSION, BURN_IN, SEED + 5000,
))

print(f"Generated {len(synthetic_animals)} synthetic animals "
      f"({N_SYNTHETIC} BE + {N_SYNTHETIC} SC)")
for a in synthetic_animals:
    p = a['true_params']
    param_str = ', '.join(f'{k}={v:.3f}' for k, v in p.items())
    print(f"  {a['animal_id']} [{a['true_model']}]: acc={a['mean_accuracy']:.2f}, {param_str}")

## 3. Grid-Search Recovery

For each synthetic animal, run the grid-search CV and check:
- Does it pick the correct model?
- How close are the recovered parameters to ground truth?

In [ ]:
grid = COARSE_GRID if QUICK else DEFAULT_GRID

gs_results = []  # one dict per synthetic animal

for sa in synthetic_animals:
    aid = sa['animal_id']
    sessions = sa['sessions']
    print(f"\n{aid} [{sa['true_model']}]...")

    be_errors, sc_errors = [], []
    be_params_list, sc_params_list = [], []

    for seed in range(1, N_CV_SEEDS + 1):
        for model_type, error_list, param_list in [
            ('BE', be_errors, be_params_list),
            ('SC', sc_errors, sc_params_list),
        ]:
            result = grid_search_cv(
                sessions, model_type, grid[model_type],
                n_folds=2, seed=seed, burn_in=BURN_IN,
            )
            error_list.append(result['avg_test_error'])
            param_list.append(result['best_params_single'])

    be_mean = np.mean(be_errors)
    sc_mean = np.mean(sc_errors)
    picked = 'BE' if be_mean < sc_mean else 'SC'
    correct = picked == sa['true_model']

    # Best params from the winning model's best seed
    if picked == 'BE':
        best_idx = np.argmin(be_errors)
        best_params = be_params_list[best_idx]
    else:
        best_idx = np.argmin(sc_errors)
        best_params = sc_params_list[best_idx]

    gs_results.append({
        'animal_id': aid,
        'true_model': sa['true_model'],
        'true_params': sa['true_params'],
        'picked_model': picked,
        'correct': correct,
        'be_mean_error': be_mean,
        'sc_mean_error': sc_mean,
        'recovered_params': best_params,
        'be_errors': be_errors,
        'sc_errors': sc_errors,
    })
    print(f"  BE={be_mean:.5f}  SC={sc_mean:.5f}  → {picked} "
          f"({'✓' if correct else '✗ WRONG'})")


In [ ]:
# Summary
gs_df = pd.DataFrame(gs_results)
n_correct = gs_df['correct'].sum()
n_total = len(gs_df)
print(f"\nGrid-search model identification: {n_correct}/{n_total} "
      f"({n_correct/n_total:.0%})")

### 3b. Grid-Search Parameter Recovery Scatter

In [ ]:
# Parameter recovery: true vs recovered for correctly-identified animals
for true_model in ['BE', 'SC']:
    sub = [r for r in gs_results
           if r['true_model'] == true_model and r['correct']]
    if len(sub) < 2:
        print(f"Not enough correct {true_model} recoveries for scatter")
        continue

    # Get param names for this model
    param_names = list(sub[0]['true_params'].keys())
    n_params = len(param_names)

    fig, axes = plt.subplots(1, n_params, figsize=(4 * n_params, 4))
    if n_params == 1:
        axes = [axes]

    for ax, pname in zip(axes, param_names):
        true_vals = [r['true_params'][pname] for r in sub]
        # Map recovered param names
        rec_vals = []
        for r in sub:
            rp = r['recovered_params']
            # Handle sigma_percep vs sigma_noise naming
            if pname in rp:
                rec_vals.append(rp[pname])
            elif pname == 'sigma_percep' and 'sigma_noise' in rp:
                rec_vals.append(rp['sigma_noise'])
            elif pname == 'sigma_noise' and 'sigma_percep' in rp:
                rec_vals.append(rp['sigma_percep'])
            else:
                rec_vals.append(np.nan)

        ax.scatter(true_vals, rec_vals, c=COLOURS[true_model], alpha=0.7, s=40)
        lims = [min(min(true_vals), min(rec_vals)) * 0.9,
                max(max(true_vals), max(rec_vals)) * 1.1]
        ax.plot(lims, lims, '--', color='grey', alpha=0.5)
        ax.set_xlabel(f'True {pname}')
        ax.set_ylabel(f'Recovered {pname}')
        ax.set_title(pname)
        ax.set_aspect('equal')

    fig.suptitle(f'{true_model}: Grid-Search Parameter Recovery (n={len(sub)})',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 4. SBI Recovery

Train SNPE with 3 different stat sets, then test parameter recovery.

In **quick mode**: trains in notebook (~2 min per network × 6 = ~12 min).
In **full mode**: loads pre-trained networks from disk.

In [ ]:
try:
    import torch
    SBI_OK = True
    print(f"torch={torch.__version__}, "
          f"{'cuda' if torch.cuda.is_available() else 'cpu'}")
except ImportError:
    SBI_OK = False
    print("SBI not available — skipping SBI sections")

In [ ]:
if SBI_OK:
    from inference.simulator import (
        create_be_simulator, create_sc_simulator,
        get_sbi_prior, wrap_for_sbi,
    )
    from inference.diagnostics import (
        parameter_recovery, run_sbc,
        plot_recovery_scatter, plot_recovery_bias,
        plot_sbc_ranks,
    )

In [ ]:
def train_or_load_snpe(model_type, stat_names, stat_label, n_sims, burn_in, seed):
    save_path = RESULTS_DIR / f'snpe_{model_type}_{stat_label}.pkl'

    n_trials = N_SESSIONS * TRIALS_PER_SESSION
    stim, cat = sample_stimuli(n_trials, 'uniform', np.random.default_rng(seed))
    creator = create_be_simulator if model_type == 'be' else create_sc_simulator
    sim = creator(stim, cat, stat_names=stat_names, burn_in=burn_in)
    prior = get_sbi_prior(sim)
    sbi_sim = wrap_for_sbi(sim)

    if not QUICK and save_path.exists():
        print(f"  Loading {save_path.name}...")
        with open(save_path, 'rb') as f:
            saved = pickle.load(f)
        return {
            **saved,
            'simulator': sim,
            'sbi_sim': sbi_sim,
            'prior': prior,
        }

    # Train
    from sbi.inference import SNPE

    print(f"  Training SNPE [{model_type.upper()}, {stat_label}] "
          f"({n_sims:,} sims)...", end=' ', flush=True)
    t0 = time.time()

    theta = prior.sample((n_sims,))
    x = torch.stack([sbi_sim(t) for t in theta])
    valid = ~torch.any(torch.isnan(x), dim=1)
    n_valid = valid.sum().item()

    inference = SNPE(prior=prior)
    inference.append_simulations(theta[valid], x[valid])
    posterior = inference.build_posterior(inference.train())

    dt = time.time() - t0
    print(f"{n_valid}/{n_sims} valid, {dt:.0f}s")

    # Save only picklable objects
    with open(save_path, 'wb') as f:
        pickle.dump({
            'posterior': posterior,
            'param_names': sim.get_param_names(),
            'stat_names': stat_names,
            'stat_label': stat_label,
            'model_type': model_type,
        }, f)

    return {
        'posterior': posterior,
        'prior': prior,
        'simulator': sim,
        'sbi_sim': sbi_sim,
        'param_names': sim.get_param_names(),
        'stat_names': stat_names,
        'stat_label': stat_label,
        'model_type': model_type,
    }

In [ ]:
# Train 6 networks: 2 models × 3 stat sets
sbi_networks = {}  # (model_type, stat_label) -> trained network data

if SBI_OK:
    stat_configs = [
        ('heuristics', HEURISTIC_STATS),
        ('um_only', UM_STATS),
        ('mixed', MIXED_STATS),
    ]

    for model_type in ['be', 'sc']:
        for stat_label, stat_names in stat_configs:
            key = (model_type, stat_label)
            sbi_networks[key] = train_or_load_snpe(
                model_type, stat_names, stat_label,
                N_SBI_SIMS, BURN_IN, SEED,
            )

    print(f"\n{len(sbi_networks)} networks ready")
else:
    print("Skipping SBI training (torch not available)")

### 4b. SBI Parameter Recovery

In [ ]:
sbi_recovery_results = {}  # (model_type, stat_label) -> recovery dict

if SBI_OK:
    for (model_type, stat_label), net in sbi_networks.items():
        print(f"\nRecovery: {model_type.upper()} / {stat_label}")
        rec = parameter_recovery(
            posterior=net['posterior'],
            simulator=net['sbi_sim'],
            prior=net['prior'],
            n_recoveries=N_SBI_RECOVERY,
            n_posterior_samples=500,
            seed=SEED,
            param_names=net['param_names'],
        )
        sbi_recovery_results[(model_type, stat_label)] = rec

        print(f"  Valid: {rec['n_valid']}/{N_SBI_RECOVERY}")
        for j, name in enumerate(rec['param_names']):
            print(f"  {name}: r={rec['correlation'][j]:.3f}, "
                  f"RMSE={rec['rmse'][j]:.4f}, "
                  f"coverage={rec['coverage_90'][j]:.2f}")

In [ ]:
# Plot recovery scatters: one figure per model, columns = stat sets
if SBI_OK:
    for model_type in ['be', 'sc']:
        fig_rows = []
        stat_labels = ['heuristics', 'um_only', 'mixed']

        for stat_label in stat_labels:
            key = (model_type, stat_label)
            if key in sbi_recovery_results:
                rec = sbi_recovery_results[key]
                fig = plot_recovery_scatter(
                    rec,
                    title=f'{model_type.upper()} — {stat_label}',
                )
                plt.show()
                plt.close(fig)

### 4c. SBC Calibration Check

In [ ]:
sbc_results = {}  # (model_type, stat_label) -> sbc dict

if SBI_OK:
    for (model_type, stat_label), net in sbi_networks.items():
        print(f"SBC: {model_type.upper()} / {stat_label}...", end=' ', flush=True)
        t0 = time.time()
        sbc = run_sbc(
            posterior=net['posterior'],
            simulator=net['sbi_sim'],
            prior=net['prior'],
            n_sbc_runs=N_SBC_RUNS,
            n_posterior_samples=500,
            seed=SEED,
            param_names=net['param_names'],
            show_progress=False,
        )
        sbc_results[(model_type, stat_label)] = sbc
        print(f"{time.time()-t0:.0f}s")

        fig = plot_sbc_ranks(sbc, title=f'{model_type.upper()} / {stat_label}')
        plt.show()
        plt.close(fig)

### 4d. SBI Model Identification

For each synthetic animal, condition both BE and SC posteriors on its data.
The model with higher marginal likelihood (or lower UM error) wins.

In [ ]:
sbi_identification = []  # per synthetic animal, per stat set

if SBI_OK:
    for stat_label in ['heuristics', 'um_only', 'mixed']:
        be_net = sbi_networks.get(('be', stat_label))
        sc_net = sbi_networks.get(('sc', stat_label))
        if be_net is None or sc_net is None:
            continue

        print(f"\nModel identification via SBI [{stat_label}]")

        for sa in synthetic_animals:
            # Pool all sessions
            all_stim, all_cat, all_ch = [], [], []
            for sess in sa['sessions']:
                arrays = sess.trials.get_arrays()
                v = ~arrays['no_response']
                all_stim.append(arrays['stimuli'][v])
                all_cat.append(arrays['categories'][v])
                all_ch.append(arrays['choices'][v])
            stim = np.concatenate(all_stim)
            cat = np.concatenate(all_cat)
            ch = np.concatenate(all_ch)

            # Compute observed stats
            stat_names = be_net['stat_names']
            obs = compute_summary_stats(
                ch, stim, cat,
                stat_names=stat_names, return_dict=False,
            )
            obs = np.nan_to_num(obs, nan=0.0)
            x_obs = torch.tensor(obs, dtype=torch.float32)

            # Log prob under each posterior
            try:
                be_samples = be_net['posterior'].sample((200,), x=x_obs)
                sc_samples = sc_net['posterior'].sample((200,), x=x_obs)

                be_lp = be_net['posterior'].log_prob(be_samples, x=x_obs).mean().item()
                sc_lp = sc_net['posterior'].log_prob(sc_samples, x=x_obs).mean().item()
            except Exception as e:
                be_lp, sc_lp = np.nan, np.nan

            picked = 'BE' if be_lp > sc_lp else 'SC'
            correct = picked == sa['true_model']

            sbi_identification.append({
                'animal_id': sa['animal_id'],
                'true_model': sa['true_model'],
                'stat_set': stat_label,
                'picked_model': picked,
                'correct': correct,
                'be_log_prob': be_lp,
                'sc_log_prob': sc_lp,
            })

        sub = [r for r in sbi_identification if r['stat_set'] == stat_label]
        n_ok = sum(r['correct'] for r in sub)
        print(f"  {stat_label}: {n_ok}/{len(sub)} correct "
              f"({n_ok/len(sub):.0%})")

    sbi_id_df = pd.DataFrame(sbi_identification)

## 5. Summary: Method Comparison

In [ ]:
print("MODEL IDENTIFICATION ACCURACY")
print("=" * 50)

# Grid-search
gs_correct = gs_df['correct'].sum()
gs_total = len(gs_df)
print(f"Grid-search CV:  {gs_correct}/{gs_total} ({gs_correct/gs_total:.0%})")
for tm in ['BE', 'SC']:
    sub = gs_df[gs_df['true_model'] == tm]
    n = sub['correct'].sum()
    print(f"  True {tm}: {n}/{len(sub)}")

# SBI
if SBI_OK and len(sbi_identification) > 0:
    print()
    for stat_label in ['heuristics', 'um_only', 'mixed']:
        sub = sbi_id_df[sbi_id_df['stat_set'] == stat_label]
        n_ok = sub['correct'].sum()
        print(f"SBI ({stat_label:>10s}): {n_ok}/{len(sub)} "
              f"({n_ok/len(sub):.0%})")
        for tm in ['BE', 'SC']:
            ss = sub[sub['true_model'] == tm]
            n = ss['correct'].sum()
            print(f"  True {tm}: {n}/{len(ss)}")

In [ ]:
# Recovery quality comparison (SBI only)
if SBI_OK and sbi_recovery_results:
    rows = []
    for (model_type, stat_label), rec in sbi_recovery_results.items():
        for j, name in enumerate(rec['param_names']):
            rows.append({
                'model': model_type.upper(),
                'stat_set': stat_label,
                'param': name,
                'correlation': rec['correlation'][j],
                'rmse': rec['rmse'][j],
                'coverage_90': rec['coverage_90'][j],
                'bias': rec['bias'][j],
            })

    rec_df = pd.DataFrame(rows)
    print("\nSBI PARAMETER RECOVERY SUMMARY")
    print("=" * 70)
    for model_type in ['BE', 'SC']:
        sub = rec_df[rec_df['model'] == model_type]
        print(f"\n{model_type}:")
        pivot = sub.pivot_table(
            index='param',
            columns='stat_set',
            values=['correlation', 'coverage_90'],
        ).round(3)
        print(pivot.to_string())

## 6. Save Results

In [ ]:
results = {
    'synthetic_animals': [
        {k: v for k, v in sa.items() if k != 'sessions'}
        for sa in synthetic_animals
    ],
    'grid_search': gs_results,
    'config': {
        'quick': QUICK, 'n_synthetic': N_SYNTHETIC,
        'n_sessions': N_SESSIONS, 'trials_per_session': TRIALS_PER_SESSION,
        'burn_in': BURN_IN, 'n_cv_seeds': N_CV_SEEDS,
        'n_sbi_sims': N_SBI_SIMS, 'n_sbi_recovery': N_SBI_RECOVERY,
    },
}

if SBI_OK:
    results['sbi_recovery'] = {
        k: {kk: vv for kk, vv in v.items() if not callable(vv)}
        for k, v in sbi_recovery_results.items()
    }
    results['sbi_identification'] = sbi_identification
    results['sbc'] = {
        k: {kk: vv for kk, vv in v.items()
            if isinstance(vv, (np.ndarray, list, str, int, float))}
        for k, v in sbc_results.items()
    }

save_path = RESULTS_DIR / f'recovery_results_{"quick" if QUICK else "full"}.pkl'
with open(save_path, 'wb') as f:
    pickle.dump(results, f)
print(f"Saved: {save_path}")

## Interpretation Guide

**Parameter recovery scatter**: points should cluster around the identity line.
If a parameter's scatter is flat (no correlation), that parameter is not
identifiable from the given stat set.

**Coverage**: 90% CI should contain the true value ~90% of the time.
Coverage ≪ 90% = overconfident posteriors. Coverage ≫ 90% = conservative.

**SBC rank histograms**: should be uniform. U-shaped = overconfident.
∩-shaped = underconfident. Skewed = biased.

**Model identification**: > 80% is good. If a method can't distinguish BE from
SC on synthetic data, it can't be trusted on real data either.

**Stat set comparison**: expect mixed ≥ UM-only ≥ heuristics for recovery quality.
If heuristics alone identify models well, the update matrix adds cost without
much benefit. If UM-only is clearly better, heuristic stats are not informative
enough.